## Section 0 - Drive Mount,Code Import, HF Login , OUT/Cache Dir

In [7]:
# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Add Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir --> out_quick 
os.makedirs('/content/drive/MyDrive/indic_synth/out_quick', exist_ok=True)
OUT = '/content/drive/MyDrive/indic_synth/out_quick'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/synthetic-data-pipeline
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 4), reused 8 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 5.09 KiB | 2.55 MiB/s, done.
From https://github.com/rahulkolayikkath/synthetic-data-pipeline
   13da2dc..919e384  main       -> origin/main
Updating 13da2dc..919e384
Fast-forward
 notebooks/colab_quick_runbook.ipynb        | 350 ++++++++++++-----------------
 src/indic_synth/tts_generation/pipeline.py |   5 +-
 2 files changed, 145 insertions(+), 210 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable f

In [8]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
# Add Cache Dir for Gemma (24gb) - only Run cell if you have enough G-drive storage
import os
#os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"
os.environ["HF_HOME"] = "/content/hf_cache"

## Section 1 - 3 (till TTS Generation)

In [ ]:
# Install for all stages till tts generation
!pip install -r requirements.txt

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages data_acquisition

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages audio_engineering

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages sentence_generation

### Section 4 - TTS Generation

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

In [ ]:
# Restart The session and Run Section 0 

In [10]:
# Check transformer Version  # -> 4.49.0
import transformers
print(transformers.__version__)

4.49.0


In [11]:
!python scripts/run.py --config config.quick.yaml --stages tts_generation

17:22:41 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
17:22:41 INFO    run | =========== stage: tts_generation ===========
2026-06-13 17:22:45.358227: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 1.444 seconds.
Prefix dict has been built successfully.
Word segmentation module jieba initialized.

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /content/hf_cache/hub/models--ai4bharat--IndicF5/snapshots/ba85abedf18dc479a447eaa0eccbd76ab78a47d5/checkpoints/vocab.txt
token :  custom
17:22:58 INFO    run | IndicF5 loaded: ai4bharat/IndicF5 on cuda
17:22:58 INFO    run | 

## Section - 5 QC 

In [ ]:
# Update the Versions back
!pip install -r requirements.txt

In [ ]:
# Restart and Run Section zero

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages quality_control

In [ ]:
print(open(f'{OUT}/qc_summary.json').read())

### Sanity Check - Rejected by QC Review

In [ ]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [ ]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:] # Edit show much you want to Verify 
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))